# Creating a script that will format columns in a way to be able to copy and paste batch variants into PolyPhen-2

In [1]:
#importing needed packages
import pandas as pd
import re
import os

In [ ]:
#uploading the parsed variant file, for AOU make sure to upload the file into the cloud folder as a CSV (File->open->upload)
parsed = "2026_B2_variants_amr_parsed_CT.csv"
df = pd.read_csv(parsed)
df.head() #removed data output to comply with All of Us data user policies

In [3]:
#amino acid dictionary, maps the amino acid 3 letter code to the corresponding letter 
aa_map = {
    "Ala":"A","Arg":"R","Asn":"N","Asp":"D","Cys":"C",
    "Glu":"E","Gln":"Q","Gly":"G","His":"H","Ile":"I",
    "Leu":"L","Lys":"K","Met":"M","Phe":"F","Pro":"P",
    "Ser":"S","Thr":"T","Trp":"W","Tyr":"Y","Val":"V"
}

In [4]:
#filter the dataframe by rows containing missense in the consequence column
missense_rows = df['consequence'].str.contains('missense', case=False, na=False)

In [5]:
#Extract the protein change from the aa_change column
#.str.replace() to remove everything before ( and after ).
#.*\( → removing everything before the (
#\).* → removing everything after the )
df.loc[missense_rows, "aa_extract"] = df.loc[missense_rows, "aa_change"].str.replace(r".*\(", "", regex=True).str.replace(r"\).*", "", regex=True)

In [6]:
#checking to see if it worked
df.loc[missense_rows, ["aa_change", "aa_extract"]].head()

,aa_change,aa_extract
277,ENSP00000369497.3:p.(Ile3Val),Ile3Val
278,ENSP00000369497.3:p.(Thr10Lys),Thr10Lys
280,ENSP00000369497.3:p.(Arg18His),Arg18His
281,ENSP00000369497.3:p.(Lys21Arg),Lys21Arg
286,ENSP00000369497.3:p.(Ile27Val),Ile27Val


In [7]:
#Creating the AA1 column with the reference amino acid
df.loc[missense_rows, "AA1"] = (df.loc[missense_rows, "aa_extract"].str.replace(r"\d+[A-Za-z]{3}", "", regex=True))
df.loc[missense_rows,["aa_extract", "AA1"]].head()

,aa_extract,AA1
277,Ile3Val,Ile
278,Thr10Lys,Thr
280,Arg18His,Arg
281,Lys21Arg,Lys
286,Ile27Val,Ile


In [8]:
#Creating the Position column
df.loc[missense_rows, "Position"] = (df.loc[missense_rows, "aa_extract"].str.replace(r"[A-Za-z]{3}", "", regex=True).str.replace(r"[A-Za-z]{3}", "", regex=True))
df.loc[missense_rows,["aa_extract", "AA1", "Position"]].head()

,aa_extract,AA1,Position
277,Ile3Val,Ile,3
278,Thr10Lys,Thr,10
280,Arg18His,Arg,18
281,Lys21Arg,Lys,21
286,Ile27Val,Ile,27


In [9]:
#Creating the AA2 column for the alternative amino acid
df.loc[missense_rows, "AA2"] = (df.loc[missense_rows, "aa_extract"].str.replace(r"^[A-Za-z]{3}\d+", "", regex=True))
df.loc[missense_rows,["aa_extract", "AA1", "Position", "AA2"]].head()

,aa_extract,AA1,Position,AA2
277,Ile3Val,Ile,3,Val
278,Thr10Lys,Thr,10,Lys
280,Arg18His,Arg,18,His
281,Lys21Arg,Lys,21,Arg
286,Ile27Val,Ile,27,Val


In [10]:
#Converting amino acids to the letter labels
df.loc[missense_rows, "Ref_AA"] = df.loc[missense_rows, "AA1"].map(aa_map)
df.loc[missense_rows, "Alt_AA"] = df.loc[missense_rows, "AA2"].map(aa_map)
df.loc[missense_rows,["aa_extract", "AA1", "Position", "AA2","Ref_AA","Alt_AA"]].head()

,aa_extract,AA1,Position,AA2,Ref_AA,Alt_AA
277,Ile3Val,Ile,3,Val,I,V
278,Thr10Lys,Thr,10,Lys,T,K
280,Arg18His,Arg,18,His,R,H
281,Lys21Arg,Lys,21,Arg,K,R
286,Ile27Val,Ile,27,Val,I,V


In [11]:
#Creating the Protein ID column
df.loc[missense_rows, "Protein"] = "P51587" #updated as needed for correct protein code. B1: P38398 , B2: P51587
df.loc[missense_rows,["aa_extract", "AA1", "Position", "AA2","Ref_AA","Alt_AA","Protein"]].head()

,aa_extract,AA1,Position,AA2,Ref_AA,Alt_AA,Protein
277,Ile3Val,Ile,3,Val,I,V,P51587
278,Thr10Lys,Thr,10,Lys,T,K,P51587
280,Arg18His,Arg,18,His,R,H,P51587
281,Lys21Arg,Lys,21,Arg,K,R,P51587
286,Ile27Val,Ile,27,Val,I,V,P51587


In [ ]:
#seeing all new columns
df.loc[missense_rows, : ].head() #data removed to comply with the All of Us data user policies.
#output is the dataframe from the full csv file, plus the 7 new columns created above

In [13]:
#exporting the updated spreadsheet
file_name = "2026_B2_variants_amr_parsed_PP2_CT.csv"
df.to_csv(file_name, index=False)